# Lung Cancer Classification

This notebook is intentionally data-dependent: it stops with an actionable error when the authorized CSV is unavailable. It does not contain fabricated metrics or predictions.

In [ ]:
import os
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 49
DATA_PATH = Path(os.environ.get("LUNG_CANCER_DATA", "data.csv")).expanduser()
if not DATA_PATH.is_absolute():
    DATA_PATH = (Path.cwd() / DATA_PATH).resolve()
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Provide the authorized CSV as data.csv "
        "or set LUNG_CANCER_DATA to its path; no training or metrics were produced."
    )


## Load and validate the data contract

Column names are normalized only; values are not imputed. Rejecting missing/unknown values avoids silently changing the experiment.

In [ ]:
def normalize_columns(frame: pd.DataFrame) -> pd.DataFrame:
    normalized = frame.copy()
    normalized.columns = (
        normalized.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "_", regex=True)
    )
    return normalized


dataset = normalize_columns(pd.read_csv(DATA_PATH))
FEATURE_COLUMNS = [
    "GENDER", "AGE", "SMOKING", "YELLOW_FINGERS", "ANXIETY", "PEER_PRESSURE",
    "CHRONIC_DISEASE", "FATIGUE", "ALLERGY", "WHEEZING", "ALCOHOL_CONSUMING",
    "COUGHING", "SHORTNESS_OF_BREATH", "SWALLOWING_DIFFICULTY", "CHEST_PAIN",
]
TARGET_COLUMN = "LUNG_CANCER"
REQUIRED_COLUMNS = FEATURE_COLUMNS + [TARGET_COLUMN]
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(dataset.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")
if dataset[REQUIRED_COLUMNS].isna().any().any():
    missing_counts = dataset[REQUIRED_COLUMNS].isna().sum()
    raise ValueError(f"Dataset contains missing values: {missing_counts[missing_counts > 0].to_dict()}")

dataset["GENDER"] = dataset["GENDER"].astype(str).str.strip().str.upper()
dataset[TARGET_COLUMN] = dataset[TARGET_COLUMN].astype(str).str.strip().str.upper()
if not set(dataset["GENDER"]).issubset({"M", "F"}):
    raise ValueError("GENDER must contain only M or F")
if not set(dataset[TARGET_COLUMN]).issubset({"YES", "NO"}):
    raise ValueError("LUNG_CANCER must contain only YES or NO")

dataset["GENDER"] = dataset["GENDER"].map({"M": 0, "F": 1})
dataset[TARGET_COLUMN] = dataset[TARGET_COLUMN].map({"NO": 0, "YES": 1})
for column in FEATURE_COLUMNS[1:]:
    dataset[column] = pd.to_numeric(dataset[column], errors="raise")
if not np.isfinite(dataset[FEATURE_COLUMNS[1:]].to_numpy(dtype=float)).all():
    raise ValueError("Numeric feature columns must contain only finite values")

X = dataset[FEATURE_COLUMNS].copy()
y = dataset[TARGET_COLUMN].copy()
class_counts = y.value_counts().sort_index()
if len(class_counts) != 2:
    raise ValueError(f"LUNG_CANCER must contain both classes; found {class_counts.to_dict()}")
if class_counts.min() < 5:
    raise ValueError("Each target class needs at least 5 rows for the configured 5-fold CV")
print(f"Loaded {len(dataset)} rows and {len(FEATURE_COLUMNS)} features from {DATA_PATH}")
print("Class counts:", class_counts.to_dict())


## Exploratory correlation (diagnostic only)

This plot is not used to select features or fit a model, so it cannot influence the held-out evaluation.

In [ ]:
correlation = dataset[FEATURE_COLUMNS + [TARGET_COLUMN]].corr(numeric_only=True)
plt.figure(figsize=(12, 9))
sns.heatmap(correlation, cbar=True, square=True, annot=False, cmap="vlag", center=0)
plt.title("Feature/target correlation (descriptive only)")
plt.tight_layout()
plt.show()


## Leakage-safe split

The test partition is held out once. Model comparison uses only the training partition through seeded stratified cross-validation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print("Train shape:", X_train.shape, "Test shape:", X_test.shape)


## Candidate models and cross-validation

Scaling is fitted inside a pipeline, separately within each training fold. The test set is not used for model selection.

In [ ]:
models = {
    "Logistic regression": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(solver="liblinear", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "KNN classifier": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier()),
    ]),
    "Decision tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=200, criterion="entropy", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(random_state=RANDOM_STATE),
    "Gradient boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for model_name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
    cv_rows.append({
        "model": model_name,
        "cv_f1_mean": scores.mean(),
        "cv_f1_std": scores.std(),
    })
cv_results = pd.DataFrame(cv_rows).sort_values(
    by=["cv_f1_mean", "model"], ascending=[False, True], ignore_index=True
)
cv_results


## Final held-out evaluation

The best model is selected by training-fold CV only, then refit on all training rows. Metrics below are produced from the untouched test partition; they are not hard-coded.

In [ ]:
best_model_name = cv_results.loc[0, "model"]
classifier = clone(models[best_model_name])
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

metrics = {
    "model": best_model_name,
    "accuracy": accuracy_score(y_test, y_pred),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
}
if hasattr(classifier, "predict_proba"):
    metrics["roc_auc"] = roc_auc_score(y_test, classifier.predict_proba(X_test)[:, 1])
print(metrics)
print(classification_report(y_test, y_pred, zero_division=0))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title(f"Held-out test confusion matrix: {best_model_name}")
plt.tight_layout()
plt.show()


## Save the trained model

The artifact is generated only after a successful data-backed run and is ignored by Git.

In [ ]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "lc.pkl"
joblib.dump(
    {
        "model": classifier,
        "feature_columns": FEATURE_COLUMNS,
        "target_column": TARGET_COLUMN,
        "random_state": RANDOM_STATE,
        "selected_by": "5-fold stratified CV on training partition",
    },
    MODEL_PATH,
)
print(f"Saved model artifact to {MODEL_PATH.resolve()}")


## Prediction API (requires caller-provided values)

No arbitrary sample row is embedded. Pass a mapping with the exact feature names and use the same encodings as the validated CSV (`GENDER`: `M`/`F`, numeric risk factors).

In [ ]:
def predict_one(feature_values: dict[str, object]):
    missing = sorted(set(FEATURE_COLUMNS) - set(feature_values))
    extra = sorted(set(feature_values) - set(FEATURE_COLUMNS))
    if missing or extra:
        raise ValueError(f"Feature keys must match the training schema; missing={missing}, extra={extra}")
    row = pd.DataFrame([feature_values], columns=FEATURE_COLUMNS)
    row["GENDER"] = row["GENDER"].astype(str).str.strip().str.upper().map({"M": 0, "F": 1})
    if row["GENDER"].isna().any():
        raise ValueError("Prediction GENDER must be M or F")
    for column in FEATURE_COLUMNS[1:]:
        row[column] = pd.to_numeric(row[column], errors="raise")
    return int(classifier.predict(row)[0])
